In [ ]:
!pip install -q openai chromadb

In [ ]:
import openai

In [ ]:
# The hard-coded API key that was here during the live class has been removed.
# NEVER paste a key into a notebook you share - it is a leaked key the moment you do.
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key: ")

openai.api_key = os.environ["OPENAI_API_KEY"]


In [ ]:
CHAT_MODEL='gpt-4o-mini'


In [ ]:
EMBEDDING_MODEL='text-embedding-3-small'

In [ ]:
question="What is our company's deploy window for production?"
chat_response=openai.chat.completions.create(model=CHAT_MODEL,
                                             messages=[{'role':'user',
                                                        'content':question}])
print(chat_response.choices[0].message.content)

I'm sorry, but I don't have access to your company's specific policies or schedules, including the deploy window for production. I recommend checking your company's internal documentation or reaching out to your development or operations team for that information.


In [ ]:
documents=[{'title': 'HireAI — Scalable MVP Architecture Plan (Free-Tier First)',
            'text': '''# HireAI — Scalable MVP Architecture Plan (Free-Tier First)

**Goal:** Ship an MVP for the first 100–1,000 users using free AI tools and free-tier infrastructure, built with Claude Code — architected so that migrating to the full paid/scaled stack (the one in `HireAI-Architecture.md`) is a config swap, not a rewrite.

---

## 1. The Core Principle: Modular Monolith + Adapter Boundaries

Do **not** build microservices, Kubernetes, or Kong for 1,000 users. Build **one deployable app** whose internal folders are the exact service boundaries you'll split along later. Every external dependency (LLM, storage, queue, telephony, email) sits behind a small interface ("adapter"), so scaling = swapping the adapter's implementation via an environment variable.

```
apps/
  web/                     → Next.js 15 (App Router) — dashboard + candidate portal + API
packages/
  db/                      → Drizzle ORM schema + migrations (Postgres)
  core/                    → business logic, organized by module (future services)
    jobs/                  → JD generation, job CRUD
    candidates/            → resume parsing, scoring, ranking
    screening/             → Round-1 screening (async text/voice)
    scheduling/            → slots, bookings, calendar sync
    interviews/            → AI interview engine (Round 2)
    analytics/             → pipeline funnel, reports
  adapters/                → ALL external dependencies behind interfaces
    llm/                   → generate(), extract(), score(), embed()
    storage/               → put(), get(), signedUrl()        (S3-compatible API)
    queue/                 → enqueue(), worker()               (pg-boss now, BullMQ later)
    email/                 → send()
    telephony/             → call(), transcribe()              (stub in MVP)
  shared/                  → types, zod schemas, auth helpers
```

**Rule that makes migration painless:** `core/*` modules never import each other's internals and never call vendors directly — only through `adapters/*` and the DB. When you outgrow the monolith, each `core/` folder becomes a service with the same interface, and each adapter gets a production implementation.

---

## 2. MVP Tech Stack (all free tiers)

| Layer | MVP (free) | Scale target (from HireAI-Architecture.md) | Migration effort |
|---|---|---|---|
| Frontend + API | Next.js 15 + TypeScript + Tailwind + shadcn/ui on **Vercel free** | Same frontend; API extracted to NestJS/Fastify pods | Low — `core/` is framework-agnostic |
| Database | **Supabase free** (Postgres 500MB + pgvector) | RDS / Supabase Pro Postgres | None — same engine, same Drizzle migrations |
| ORM | Drizzle (SQL-first, no runtime engine) | Same | None |
| Auth | **Supabase Auth** (email + Google OAuth, free) | Same, or JWT/OAuth2 service | None until enterprise SSO |
| Vector search | **pgvector** in same Postgres | pgvector on bigger instance (Pinecone only if >10M vectors) | None |
| File storage | **Supabase Storage** (1GB free, S3-compatible) | AWS S3 | Env-var swap — same S3 API in adapter |
| Background jobs | **pg-boss** (queue inside Postgres — zero extra infra) | Redis + BullMQ | Adapter swap; job payloads identical |
| Cache | None needed at this scale (Postgres is fast enough) | Redis / Upstash | Add later behind adapter |
| Full-text search | Postgres `tsvector` | Elasticsearch/Typesense if needed | Deferred; likely never needed |
| Realtime (live transcript) | Supabase Realtime channels | Socket.io / managed WS | Adapter swap |
| Email | Resend free (3k/mo) or Brevo free (300/day) | SendGrid/SES | Adapter swap |
| Monitoring | Sentry free + Vercel analytics | Sentry + Datadog + PostHog | Additive |
| Billing | Skip in MVP (manual invoices) | Stripe | Additive module |

**Why Supabase as the spine:** one free account gives Postgres + pgvector + Auth + Storage + Realtime — the entire data layer of the target architecture, with a direct paid upgrade path and no vendor rewrite (it's plain Postgres and S3 APIs underneath).

---

## 3. Free AI Stack (the important part)

All AI calls go through **one adapter**: `adapters/llm` exposing `generate()`, `extractStructured()`, `scoreAgainstRubric()`, `embed()`. Provider chosen by env var, with automatic fallback chain when a free tier rate-limits.

| Capability | Free option (MVP) | Notes / limits | Scale swap |
|---|---|---|---|
| JD generation, scoring, question gen | **Google Gemini 2.5 Flash** free tier | ~1,500 req/day free — plenty for 1k users | Claude Sonnet/Opus via API key swap |
| Fallback LLM | **Groq** (Llama 3.3 70B) free tier | Very fast; good for scoring/extraction | Same |
| Structured resume extraction | Gemini Flash with JSON schema output | pdf-parse / mammoth for text extraction first (free, local) | Claude with tool-use |
| Embeddings (candidate↔JD matching) | **Gemini `text-embedding-004`** (free) or local `bge-small` via transformers.js | Stored in pgvector | OpenAI/Voyage embeddings |
| Speech-to-text | **Groq Whisper large-v3** (free tier) | For uploaded audio answers | OpenAI Whisper / Deepgram |
| Text-to-speech | **Edge-TTS** (free, unofficial MS voices) or browser SpeechSynthesis | Good enough for async interviews | ElevenLabs / OpenAI TTS |
| Dev-time AI | **Claude Code** | Builds/refactors the codebase itself | — |

**Rate-limit strategy:** every LLM call goes through the queue with retry + provider fallback (Gemini → Groq → defer). At 1,000 users this stays inside free tiers because the heavy AI work (parsing, scoring) is per-candidate, batched, and cached (never re-score an unchanged resume — hash the resume+JD pair).

---

## 4. The One Big MVP Substitution: Voice → Async

Twilio outbound IVR calls are the most expensive, most complex, least-free part of the target architecture. For the MVP, **replace live phone screening with an async web-based screening**, which keeps the exact same data model (`screening_calls` table: transcript, scores, recording URL):

- **Round 1 (MVP):** Candidate gets an email/WhatsApp link → opens a mobile web page → answers 5–8 questions by **recording audio in the browser** (MediaRecorder API, free) → uploads to storage → queue job runs Groq Whisper STT → LLM scores against rubric → auto-advance past threshold.
- **Round 2 (MVP):** Same page, but conversational: TTS asks a question (Edge-TTS/browser), candidate answers by voice, STT + LLM generate an adaptive follow-up. This is the full "AI interview" experience with zero telephony cost.
- **Scale:** implement `adapters/telephony` with Twilio + streaming STT — the screening module's inputs/outputs (transcript in, scores out) don't change at all.

This is the design decision that makes a free MVP possible without repainting yourself into a corner.

---

## 5. Data Model (multi-tenant from day 1)

Use the schema already defined in `HireAI-Architecture.md` §5 almost verbatim — it's good. Non-negotiables to bake in now because they're brutal to retrofit:

1. **`org_id` on every table** + Postgres **Row-Level Security** (Supabase makes this native). Multi-tenancy is the #1 thing you cannot bolt on later.
2. **UUIDs for all primary keys** (no sequential int leaks, safe for future sharding).
3. **Status fields as explicit state machines** (`candidate.status`: `uploaded → parsed → scored → screening_invited → screening_done → interview_scheduled → interview_done → shortlisted / rejected`). The whole pipeline automation hangs off these transitions.
4. **`events` append-only table** (org_id, entity, entity_id, event_type, payload, created_at) — this gives you analytics, audit trail, and debugging for free, and becomes your event bus when you later move to real queues/services.
5. **All AI outputs stored as `jsonb` with the prompt version** (`ai_scores: {version: "v3", ...}`) so you can re-score when prompts improve and audit for EEOC/bias compliance.

---

## 6. Phase Plan (each phase ships usable value)

**Phase 0 — Foundation (week 1)**
Monorepo (pnpm + Turborepo), Next.js app, Supabase project, Drizzle schema + RLS, Supabase Auth (email + Google), org/user onboarding, CI via GitHub Actions (free), Sentry. Deploy to Vercel on day 1 — deploy continuously from here.

**Phase 1 — Jobs + JD Generation (week 2)**
Job CRUD, LLM adapter with Gemini/Groq fallback, JD generator with templates, skills extraction into structured fields. *First demoable value.*

**Phase 2 — Resume Pipeline (weeks 3–4)**
Bulk upload (drag-drop → Supabase Storage), pg-boss queue, text extraction (pdf-parse/mammoth), LLM structured extraction, embedding + pgvector similarity, weighted scoring vs JD, ranked candidate list with threshold filter. *This is the core product — most of the recommendation-engine work lives here.*

**Phase 3 — Async Screening, Round 1 (weeks 5–6)**
Candidate-facing mobile web page, browser audio recording, Whisper STT via Groq, rubric scoring, auto-advance on threshold, recruiter review UI with playback + transcript.

**Phase 4 — Scheduling (week 7)**
Recruiter availability slots, candidate self-serve booking page, Google Calendar API (free), email notifications via Resend. Skip Outlook until a customer asks.

**Phase 5 — AI Interview, Round 2 (weeks 8–9)**
Conversational async interview: TTS question → voice answer → adaptive follow-up, dimensional scoring, report generation. Reuses everything from Phase 3.

**Phase 6 — Pipeline Dashboard + Handoff (week 10)**
Funnel view per job, candidate comparison, one-click Round-3 scheduling, org analytics off the `events` table. Then: beta users.

**Deferred until revenue:** Twilio live IVR, Outlook/MS Graph, Stripe billing, Elasticsearch, Redis, SOC 2 tooling.

---

## 7. Migration Playbook (MVP → Scale, "starting tomorrow")

Triggers and the exact move for each — no rewrites, because of the adapter boundaries:

| Pain signal | Move | Why it's cheap |
|---|---|---|
| Free LLM rate limits hit | Set `LLM_PROVIDER=anthropic`, add API key | Same adapter interface |
| Supabase 500MB DB full | Upgrade Supabase plan or `pg_dump` → RDS | Plain Postgres |
| Queue jobs backing up | Swap pg-boss impl → BullMQ + Upstash Redis in `adapters/queue` | Same job payloads |
| Storage >1GB | Point storage adapter at S3 | Same S3 API |
| Vercel function timeouts on long AI jobs | Move worker process to Railway/Fly.io container (queue consumer already runs standalone) | Worker is already a separate entrypoint |
| Need live phone screening (paying customers) | Implement `adapters/telephony` with Twilio; screening module unchanged | Transcript-in/scores-out contract |
| API load / team growth | Extract `core/candidates` (heaviest) into its own service first; `events` table becomes the integration point | Module boundaries = service boundaries |
| Enterprise deals | Add Stripe, SSO (SAML via WorkOS), SOC 2 logging on the `events` table | Additive |

**Monthly cost: $0 for the MVP phase.** First paid line items, in order, will be: LLM API (~$50–200), Supabase Pro ($25), a worker container (~$5–10).

---

## 8. Claude Code Working Method

- Keep this file + a `CLAUDE.md` (conventions: adapter rule, RLS rule, state-machine statuses) at repo root so every session has the architecture in context.
- Build phase by phase; each phase = one milestone with its own branch and deployed preview.
- Have Claude Code write integration tests for each `core/` module against a local Postgres (Docker) — these tests are your safety net for every adapter swap during migration.
- Use zod schemas in `packages/shared` as the single source of truth for API contracts — when modules become services, the contracts already exist.
'''}]

In [ ]:
print('Document', len(documents))

Document 1


In [ ]:
print('Characters', len(documents[0]['text']))

Characters 11600


In [ ]:
def chunk_documents(doc, max_size=350):
  chunks=[]
  section='Introduction'
  buffer=''

  for line in doc['text'].split('\n'):
    is_heading=line.startswith('#')
    is_full=len(buffer)+len(line)>max_size

    if buffer.strip()!='' and (is_heading or is_full):
      chunks.append({'id':doc['title']+'#'+str(len(chunks)),
                     'title':doc['title']+'>'+section,
                     'text':buffer.strip()})
      buffer=''
    if is_heading:
      section=line.replace('#','').strip()
    else:
      buffer=buffer+line+'\n'


  if buffer.strip()!='':
    chunks.append({'id':doc['title']+'#'+str(len(chunks)),
                     'title':doc['title']+'>'+section,
                     'text':buffer.strip()})

  return chunks



In [ ]:
chunks=[]
for doc in documents:
  chunks=chunks+chunk_documents(doc)
print('Total Chunks', len(chunks))
print()

for c in chunks:
  print(len(c['text']),"chars|", c['title'])

Total Chunks 40

262 chars| HireAI — Scalable MVP Architecture Plan (Free-Tier First)>HireAI — Scalable MVP Architecture Plan (Free-Tier First)
368 chars| HireAI — Scalable MVP Architecture Plan (Free-Tier First)>1. The Core Principle: Modular Monolith + Adapter Boundaries
317 chars| HireAI — Scalable MVP Architecture Plan (Free-Tier First)>1. The Core Principle: Modular Monolith + Adapter Boundaries
296 chars| HireAI — Scalable MVP Architecture Plan (Free-Tier First)>1. The Core Principle: Modular Monolith + Adapter Boundaries
348 chars| HireAI — Scalable MVP Architecture Plan (Free-Tier First)>1. The Core Principle: Modular Monolith + Adapter Boundaries
138 chars| HireAI — Scalable MVP Architecture Plan (Free-Tier First)>1. The Core Principle: Modular Monolith + Adapter Boundaries
314 chars| HireAI — Scalable MVP Architecture Plan (Free-Tier First)>1. The Core Principle: Modular Monolith + Adapter Boundaries
283 chars| HireAI — Scalable MVP Architecture Plan (Free-Tier First)>2. MVP 

# **Step-4 Embedding**

In [ ]:
def embed_texts(texts):
  response=openai.embeddings.create(model=EMBEDDING_MODEL, input=texts)
  return [item.embedding for item in response.data]
test=embed_texts(['How do I do rollback the deployment?'])


In [ ]:
test

[[0.01081085205078125,
  0.0203094482421875,
  0.04345703125,
  -0.02142333984375,
  -0.0506591796875,
  -0.0362548828125,
  -0.012054443359375,
  -0.0107574462890625,
  -0.00424957275390625,
  -0.0220794677734375,
  0.0283660888671875,
  -0.03607177734375,
  -0.006885528564453125,
  -0.0237884521484375,
  0.033416748046875,
  0.0015764236450195312,
  -0.01239776611328125,
  -0.0023403167724609375,
  -0.04302978515625,
  0.04559326171875,
  -0.04425048828125,
  0.007480621337890625,
  0.01171875,
  0.01235198974609375,
  -0.024932861328125,
  -0.00096893310546875,
  0.01465606689453125,
  -0.035003662109375,
  -0.0113372802734375,
  -0.0015935897827148438,
  0.0227508544921875,
  -0.0257415771484375,
  0.01218414306640625,
  -0.01313018798828125,
  -0.0061492919921875,
  -0.00814056396484375,
  0.0263519287109375,
  -0.0157012939453125,
  0.049560546875,
  0.0252532958984375,
  0.006916046142578125,
  -0.00472259521484375,
  -0.0360107421875,
  0.0005273818969726562,
  0.01459503173828

In [ ]:
import numpy as np

def similarity(a,b):
  a=np.array(a)
  b=np.array(b)
  return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))




# **SToring the chunks in Vector Database- CHromaDB**

In [ ]:
import chromadb

chroma_client=chromadb.Client()

collection=chroma_client.get_or_create_collection(name='docs',metadata={"hnsw:space":"cosine"})

collection.add(ids=[c["id"]for c in chunks], embeddings=embed_texts([c['text'] for c in chunks]),
               documents=[c['text'] for c in chunks],
               metadatas=[{'title':c['title']} for c in chunks]
               )

# **Search Feature**

# **Embed the question then ask CHromaDB for the closest chunks **

In [ ]:
def search(question, n=3, where=None):
  r=collection.query(query_embeddings=embed_texts([question]),
                     n_results=n,
                     where=where)
  return [{'title':meta['title'], 'text':text, 'score':1-dist}
          for meta, text, dist in zip(r['metadatas'][0],
                                      r['documents'][0],
                                      r['distances'][0])
  ]

In [ ]:
questions=['how do we handle phone calls in the MVP?']

for q in questions:
  print(q)
  for hit in search(q, n=3):
    print(hit['score'])
    print (hit['title'])
    print (hit['text'])


how do we handle phone calls in the MVP?
0.5520776510238647
HireAI — Scalable MVP Architecture Plan (Free-Tier First)>1. The Core Principle: Modular Monolith + Adapter Boundaries
telephony/             → call(), transcribe()              (stub in MVP)
  shared/                  → types, zod schemas, auth helpers
```
0.4984699487686157
HireAI — Scalable MVP Architecture Plan (Free-Tier First)>4. The One Big MVP Substitution: Voice → Async
Twilio outbound IVR calls are the most expensive, most complex, least-free part of the target architecture. For the MVP, **replace live phone screening with an async web-based screening**, which keeps the exact same data model (`screening_calls` table: transcript, scores, recording URL):
0.4639025926589966
HireAI — Scalable MVP Architecture Plan (Free-Tier First)>4. The One Big MVP Substitution: Voice → Async
- **Scale:** implement `adapters/telephony` with Twilio + streaming STT — the screening module's inputs/outputs (transcript in, scores out) don't

In [ ]:
system_prompt='''you answer questions about internal technical documents
Rules:
1. Answer ONLY USING document. Dont use any other knowledge.
2. If the document do not contain any answeer reply ' I dont know based on document provided'
3. Be short and clear'''


def ask(question, n=3):
  hits = search(question, n=n)

  # Construct user_prompt from the question and the relevant document hits
  user_prompt = question + "\n\nRelevant Chunks:\n"
  for i, hit in enumerate(hits):
    user_prompt += f"Document {i+1}: {hit['title']}\n{hit['text']}\n\n"

  response=openai.chat.completions.create(model=CHAT_MODEL, messages=[{'role':'system',
                                                                       'content':system_prompt},
                                                                      {'role':'user',
                                                                       'content':user_prompt}
                                                                      ], temperature=0)
  answer=response.choices[0].message.content

  print(answer)


In [ ]:
ask('Which database should the MVP use and why it was chosen?')

The MVP should use **Supabase free** (Postgres 500MB + pgvector) as the database. It was chosen because it is part of the tech stack that utilizes free tiers, aligning with the goal of shipping an MVP for the first 100–1,000 users using free AI tools and infrastructure.


In [ ]:
ask('Where  is the application live?')

I don't know based on document provided.
